# Get list of entrants

In [16]:
tournament_slug = 'aav'
tournament_slug = 'allston-allstars-vi-1'
tournament_slug = 'ngpr'
event_name = 'Rishi\'s Jungle Jam'
event_name = 'Old School Melee'
event_name = 'Three Character Iron Man'
event_name = 'Melee Redemption'
event_name = 'Melee Singles'

In [17]:
%%bash
source ~/.bashrc

In [18]:
import os
import json
from gql import gql, Client
from gql.transport.requests import RequestsHTTPTransport

auth_token = os.environ['SMASHGG_TOKEN']
api_version = 'alpha'

transport = RequestsHTTPTransport(
    url=f'https://api.start.gg/gql/{api_version}',
    headers={'Authorization': f'Bearer {auth_token}'},
    use_json=True,
)

client = Client(transport=transport, fetch_schema_from_transport=False)

In [19]:
# Define the API endpoint and query for tournament details
query = gql("""
query TournamentQuery($slug: String!) {
    tournament(slug: $slug) {
        id
        name
        city
        state
        countryCode
        startAt
        endAt
        events {
            id
            videogame {
                id
                name
            }
            name
            numEntrants
        }
    }
}
""")

variables = {"slug": tournament_slug}

# Execute the query
try:
        tournament_result = client.execute(query, variable_values=variables)
        print(json.dumps(tournament_result, indent=2))
except Exception as e:
        print("Error fetching tournament details:", e)

{
  "tournament": {
    "id": 893520,
    "name": "New Game Plus Revival 9.21",
    "city": "Boston",
    "state": 3,
    "countryCode": "US",
    "startAt": 1774388700,
    "endAt": 1774411140,
    "events": [
      {
        "id": 1586819,
        "videogame": {
          "id": 1,
          "name": "Super Smash Bros. Melee"
        },
        "name": "Melee Redemption",
        "numEntrants": 0
      },
      {
        "id": 1586818,
        "videogame": {
          "id": 1,
          "name": "Super Smash Bros. Melee"
        },
        "name": "Melee Singles",
        "numEntrants": 36
      },
      {
        "id": 1586821,
        "videogame": {
          "id": 1,
          "name": "Super Smash Bros. Melee"
        },
        "name": "Melee Doubles",
        "numEntrants": 7
      },
      {
        "id": 1586820,
        "videogame": {
          "id": 33602,
          "name": "Project+"
        },
        "name": "Project + Singles",
        "numEntrants": 4
      }
    ]
  }
}


In [20]:
# Extract the event id for "Melee Singles"
events = tournament_result['tournament']['events']
melee_singles_id = next((event['id'] for event in events if event_name in event['name']), None)
print(f"{event_name} Event ID: {melee_singles_id}")

Melee Singles Event ID: 1586818


In [21]:
# Query to get entrants for Melee Singles
entrants_query = gql("""
query EventEntrants($eventId: ID!) {
    event(id: $eventId) {
        entrants(query: {page: 1, perPage: 512}) {
            nodes {
                id
                name
                participants {
                    gamerTag
                    user {
                        slug
                    }
                }
            }
        }
    }
}
""")

entrants_variables = {"eventId": melee_singles_id}

try:
        entrants_result = client.execute(entrants_query, variable_values=entrants_variables)
        print(json.dumps(entrants_result, indent=2))
except Exception as e:
        print("Error fetching entrants:", e)

{
  "event": {
    "entrants": {
      "nodes": [
        {
          "id": 23010981,
          "name": "Bigs",
          "participants": [
            {
              "gamerTag": "Bigs",
              "user": {
                "slug": "user/3b2098c3"
              }
            }
          ]
        },
        {
          "id": 23010976,
          "name": "MOOP",
          "participants": [
            {
              "gamerTag": "MOOP",
              "user": {
                "slug": "user/0d9573f9"
              }
            }
          ]
        },
        {
          "id": 23010849,
          "name": "hc | keke",
          "participants": [
            {
              "gamerTag": "keke",
              "user": {
                "slug": "user/4864af7d"
              }
            }
          ]
        },
        {
          "id": 23010836,
          "name": "RoccoChief",
          "participants": [
            {
              "gamerTag": "RoccoChief",
              "user": {
      

In [22]:
import polars as pl

# Extract entrant data and flatten participants' gamerTags

entrant_nodes = entrants_result['event']['entrants']['nodes']
data = []
for entrant in entrant_nodes:
    entrant_id = entrant['id']
    entrant_name = entrant['name']
    # There may be multiple participants per entrant; join their gamerTags with comma
    gamer_tags = [p['gamerTag'] for p in entrant.get('participants', [])]
    gamer_tag = ', '.join(gamer_tags)
    data.append({'id': entrant_id, 'name': entrant_name, 'gamertag': gamer_tag})

entrants = pl.DataFrame(data)
entrants

id,name,gamertag
i64,str,str
23010981,"""Bigs""","""Bigs"""
23010976,"""MOOP""","""MOOP"""
23010849,"""hc | keke""","""keke"""
23010836,"""RoccoChief""","""RoccoChief"""
23010778,"""hc | Good1ife""","""Good1ife"""
23010630,"""DrillYouu""","""DrillYouu"""
23010623,"""heylayow""","""heylayow"""
23010558,"""hc | PajamaPal""","""PajamaPal"""
23010442,"""RhodeIslandBean""","""RhodeIslandBean"""


In [23]:
entrant_gamertags = entrants.select(pl.col('gamertag')).to_series().to_list()

# collect seeding

In [24]:
player_ratings = pl.read_parquet('data/player-ratings.parquet')

In [25]:
pl.Config(tbl_rows=100)
entrant_ratings = (
    entrants
    .with_columns(pl.col('gamertag'))
    .join(
        player_ratings
            .with_columns(pl.col('tag').str.split(' | ').list.last()),
        how='full',
        left_on='gamertag',
        right_on='tag',
    )
    #.filter(
    #    (pl.col('tag').str.to_lowercase() == pl.col('gamertag').str.to_lowercase()) |
    #    pl.col('gamertag').is_null()
    #)
    .filter(pl.col('gamertag').is_not_null())
    .sort('rating', descending=True)
    .group_by('id').first()
    .fill_null(0)
    .sort('rating', descending=True)
)
display(entrant_ratings)

id,name,gamertag,url,rating,tag
i64,str,str,str,f64,str
23008215,"""TSI | coffee""","""coffee""","""/league/nemelee/player/25479C6…",40.230426,"""coffee"""
23009063,"""Ant""","""Ant""","""/league/nemelee/player/5DBCC34…",34.396759,"""Ant"""
23010778,"""hc | Good1ife""","""Good1ife""","""/league/nemelee/player/396A6F1…",32.577395,"""Good1ife"""
23008184,"""BRamZ""","""BRamZ""","""/league/nemelee/player/8CB30ED…",30.096264,"""BRamZ"""
23009323,"""BonkCushy""","""BonkCushy""","""/league/nemelee/player/1056255…",29.72562,"""BonkCushy"""
23009673,"""Motobug""","""Motobug""","""/league/nemelee/player/E12030B…",29.643181,"""Motobug"""
22991834,"""zaubermaus""","""zaubermaus""","""/league/nemelee/player/66DEE19…",28.995816,"""zaubermaus"""
23010849,"""hc | keke""","""keke""","""/league/nemelee/player/ACFD51E…",28.725307,"""keke"""
23008069,"""Qwerty""","""Qwerty""","""/league/nemelee/player/19B79D7…",28.154013,"""Qwerty"""


In [26]:
entrant_seeding = (
    entrant_ratings
    .sort('rating', descending=True)
    .with_row_index('seed_num')
    .with_columns(pl.col('seed_num') + 1)
)
entrant_seeding

seed_num,id,name,gamertag,url,rating,tag
u32,i64,str,str,str,f64,str
1,23008215,"""TSI | coffee""","""coffee""","""/league/nemelee/player/25479C6…",40.230426,"""coffee"""
2,23009063,"""Ant""","""Ant""","""/league/nemelee/player/5DBCC34…",34.396759,"""Ant"""
3,23010778,"""hc | Good1ife""","""Good1ife""","""/league/nemelee/player/396A6F1…",32.577395,"""Good1ife"""
4,23008184,"""BRamZ""","""BRamZ""","""/league/nemelee/player/8CB30ED…",30.096264,"""BRamZ"""
5,23009323,"""BonkCushy""","""BonkCushy""","""/league/nemelee/player/1056255…",29.72562,"""BonkCushy"""
6,23009673,"""Motobug""","""Motobug""","""/league/nemelee/player/E12030B…",29.643181,"""Motobug"""
7,22991834,"""zaubermaus""","""zaubermaus""","""/league/nemelee/player/66DEE19…",28.995816,"""zaubermaus"""
8,23010849,"""hc | keke""","""keke""","""/league/nemelee/player/ACFD51E…",28.725307,"""keke"""
9,23008069,"""Qwerty""","""Qwerty""","""/league/nemelee/player/19B79D7…",28.154013,"""Qwerty"""


# Optimize seeding to avoid repeat matchups
Swaps similarly-rated players within tiers to separate frequent opponents in the bracket.
Recency-weighted: recent matchups penalized more (90-day half-life by default).

In [27]:
from seeding_utils import build_matchup_matrix, optimize_seeding, format_swap_report

matches = pl.read_csv('data/matches.csv')

# Build recency-weighted matchup matrix for entrants
entrant_urls = entrant_seeding.filter(pl.col('url').is_not_null()).select('url').to_series().to_list()
matchup_matrix = build_matchup_matrix(matches, entrant_urls, half_life_days=90)

print(f"Matchup pairs found: {len(matchup_matrix)}")

# Show top-10 most frequent pairings among entrants
top_matchups = sorted(matchup_matrix.items(), key=lambda x: -x[1])[:20]
url_to_tag = dict(zip(
    entrant_seeding.select('url').to_series().to_list(),
    entrant_seeding.select('gamertag').to_series().to_list()
))
print("\nTop matchups among entrants:")
for (a, b), w in top_matchups:
    print(f"  {url_to_tag.get(a, a)} vs {url_to_tag.get(b, b)}: {w:.2f}")

Matchup pairs found: 241

Top matchups among entrants:
  Qwerty vs Frootbat: 3.17
  BonkCushy vs Qwerty: 3.06
  Babs vs ren: 3.03
  Ant vs zaubermaus: 2.63
  RhodeIslandBean vs Exarch: 2.61
  Byrne vs ren: 2.59
  Jeep God vs Cheezboom: 2.51
  BonkCushy vs keke: 2.40
  Byrne vs Frootbat: 2.40
  Cheezboom vs keke: 2.35
  zaubermaus vs Byrne: 2.30
  zaubermaus vs keke: 2.22
  Good1ife vs Jeep God: 2.15
  Cheezboom vs Exarch: 2.13
  Babs vs Byrne: 2.12
  Duffy vs ren: 2.06
  Qwerty vs Duffy: 2.03
  Jeep God vs zaubermaus: 2.01
  coffee vs BRamZ: 1.95
  BonkCushy vs zaubermaus: 1.93


In [28]:
# Run the optimizer
optimized_seeding, swaps, final_cost = optimize_seeding(
    entrant_seeding,
    matchup_matrix,
    url_col='url',
    seed_col='seed_num',
    max_seed_distance=6,
)

# Show swap summary
print(format_swap_report(swaps, entrant_seeding, url_col='url', tag_col='gamertag'))
print(f"\nFinal cost: {final_cost:.2f}")

Swaps made: 24
  Seed 3 ↔ 4: BRamZ ↔ Good1ife (cost −1.66)
  Seed 5 ↔ 6: Motobug ↔ BonkCushy (cost −2.56)
  Seed 7 ↔ 8: keke ↔ zaubermaus (cost −1.08)
  Seed 9 ↔ 10: Bigs ↔ Qwerty (cost −3.25)
  Seed 9 ↔ 12: Frootbat ↔ Bigs (cost −3.76)
  Seed 11 ↔ 15: Babs ↔ Jeep God (cost −1.32)
  Seed 15 ↔ 16: Byrne ↔ Jeep God (cost −3.25)
  Seed 17 ↔ 18: Duffy ↔ sfy bees (cost −0.99)
  Seed 17 ↔ 22: Puunk_ ↔ Duffy (cost −1.10)
  Seed 18 ↔ 22: Duffy ↔ sfy bees (cost −0.08)
  Seed 20 ↔ 21: GuitarLord ↔ ren (cost −2.28)
  Seed 20 ↔ 22: sfy bees ↔ GuitarLord (cost −0.57)
  Seed 23 ↔ 29: Exarch ↔ RhodeIslandBean (cost −0.49)
  Seed 24 ↔ 25: Robert WIlliwinkle III ↔ PajamaPal (cost −0.15)
  Seed 26 ↔ 32: Coffee ↔ CrucialCrazy (cost −4.67)
  Seed 27 ↔ 29: RhodeIslandBean ↔ joe chemo (cost −0.20)
  Seed 5 ↔ 7: keke ↔ Motobug (cost −0.18)
  Seed 9 ↔ 10: Qwerty ↔ Frootbat (cost −5.14)
  Seed 24 ↔ 27: RhodeIslandBean ↔ Robert WIlliwinkle III (cost −0.00)
  Seed 25 ↔ 27: Robert WIlliwinkle III ↔ PajamaPal (cos

In [29]:
# Show before/after comparison for changed seeds
changed = (
    optimized_seeding
    .filter(pl.col('seed_num') != pl.col('original_seed'))
    .select('gamertag', 'original_seed', 'seed_num', 'rating')
)
print(f"Seeds changed: {changed.height} / {entrant_seeding.height}")
changed

Seeds changed: 24 / 36


gamertag,original_seed,seed_num,rating
str,u32,i64,f64
"""BRamZ""",4,3,30.096264
"""Good1ife""",3,4,32.577395
"""keke""",8,5,28.725307
"""BonkCushy""",5,6,29.72562
"""Motobug""",6,7,29.643181
"""zaubermaus""",7,8,28.995816
"""Frootbat""",12,10,27.334145
"""Babs""",15,11,26.626799
"""Bigs""",10,12,27.883786


In [30]:
# Use optimized seeding for export (replaces entrant_seeding downstream)
entrant_seeding = optimized_seeding
entrant_seeding

seed_num,id,name,gamertag,url,rating,tag,original_seed
i64,i64,str,str,str,f64,str,u32
1,23008215,"""TSI | coffee""","""coffee""","""/league/nemelee/player/25479C6…",40.230426,"""coffee""",1
2,23009063,"""Ant""","""Ant""","""/league/nemelee/player/5DBCC34…",34.396759,"""Ant""",2
3,23008184,"""BRamZ""","""BRamZ""","""/league/nemelee/player/8CB30ED…",30.096264,"""BRamZ""",4
4,23010778,"""hc | Good1ife""","""Good1ife""","""/league/nemelee/player/396A6F1…",32.577395,"""Good1ife""",3
5,23010849,"""hc | keke""","""keke""","""/league/nemelee/player/ACFD51E…",28.725307,"""keke""",8
6,23009323,"""BonkCushy""","""BonkCushy""","""/league/nemelee/player/1056255…",29.72562,"""BonkCushy""",5
7,23009673,"""Motobug""","""Motobug""","""/league/nemelee/player/E12030B…",29.643181,"""Motobug""",6
8,22991834,"""zaubermaus""","""zaubermaus""","""/league/nemelee/player/66DEE19…",28.995816,"""zaubermaus""",7
9,23008069,"""Qwerty""","""Qwerty""","""/league/nemelee/player/19B79D7…",28.154013,"""Qwerty""",9


# Export seeding to start gg

In [56]:
# Query to get phase IDs for Melee Singles event
phases_query = gql("""
query EventPhases($eventId: ID!) {
    event(id: $eventId) {
        phases {
            id
            name
            numSeeds
        }
    }
}
""")

phases_variables = {"eventId": melee_singles_id}

try:
        phases_result = client.execute(phases_query, variable_values=phases_variables)
        print(json.dumps(phases_result, indent=2))
except Exception as e:
        print("Error fetching phases:", e)

phase_id = phases_result['event']['phases'][0]['id']
print(f'phase id: {phase_id}')

{
  "event": {
    "phases": [
      {
        "id": 2231455,
        "name": "Bracket",
        "numSeeds": 34
      }
    ]
  }
}
phase id: 2231455


In [57]:
# ...existing code...

# 1. Fetch seeds for the phase
seeds_query = gql("""
query PhaseSeeds($phaseId: ID!) {
  phase(id: $phaseId) {
    seeds(query: {perPage: 512}) {
      nodes {
        id
        entrant {
          id
        }
      }
    }
  }
}
""")
seeds_result = client.execute(seeds_query, variable_values={"phaseId": phase_id})
seed_nodes = seeds_result["phase"]["seeds"]["nodes"]
entrantid_to_seedid = {seed["entrant"]["id"]: seed["id"] for seed in seed_nodes}

# 2. Prepare seed mapping using correct seedId
seed_mapping = []
for row in entrant_seeding.iter_rows(named=True):
    entrant_id = int(row["id"])
    seed_id = entrantid_to_seedid.get(entrant_id)
    if seed_id:
        seed_mapping.append({
            "seedId": seed_id,
            "seedNum": row["seed_num"],
        })
    else:
        print(f"Warning: No seed found for entrant {entrant_id}")

print(f"Importing {len(seed_mapping)} seeds to phase {phase_id}...")

Importing 34 seeds to phase 2231455...


In [58]:
entrant_seeding = (
    entrant_seeding
    .with_columns(pl.col('id').map_elements(lambda x: entrantid_to_seedid.get(x, None)).alias('seed_id'))
)

/tmp/ipykernel_39283/1705157941.py:3: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(pl.col('id').map_elements(lambda x: entrantid_to_seedid.get(x, None)).alias('seed_id'))


In [59]:
(
    player_ratings
    .select(pl.col('tag'))
)

tag
str
"""Nouns | Aklo"""
"""DTL | 2Saint"""
"""RedBull GG IFM | aMSa"""
"""Spark"""
"""SDJ"""
"""FLY | Jmook"""
"""MATE | Kalvar"""
"""Epoodle"""
"""S3C | RapMonster"""


In [38]:
# Prepare seed mapping from entrant_ratings for the phase
seed_mapping = []
for row in entrant_seeding.iter_rows(named=True):
    seed_mapping.append({
        "seedId": row["seed_id"],
        "seedNum": row['seed_num'],
    })

print(f"Importing {len(seed_mapping)} seeds to phase {phase_id}...")

mutation = gql("""
mutation UpdatePhaseSeeding($phaseId: ID!, $seedMapping: [UpdatePhaseSeedInfo]!) {
  updatePhaseSeeding(phaseId: $phaseId, seedMapping: $seedMapping) {
    id
  }
}
""")

params = {
    "phaseId": phase_id,
    "seedMapping": seed_mapping,
}

try:
    result = client.execute(mutation, variable_values=params)
    print('Success!')
    print(result)
except Exception as e:
    print('Error:', e)

Importing 22 seeds to phase 2231455...
Success!
{'updatePhaseSeeding': {'id': 2231455}}


# view previous tournaments by player

In [39]:
player = 'hotdaniel'


In [40]:
from IPython.display import display, HTML

def display_side_by_side(*args, titles=('',)):
    html_str = ''
    if len(titles) > 0:
        html_str += '<div style="display:flex">'
    for df, title in zip(args, titles + ('',) * (len(args) - len(titles))):
        html_str += '<div style="margin-right:20px">'
        if title:
            html_str += f'<h2>{title}</h2>'
        html_str += df.to_html()
        html_str += '</div>'
    html_str += '</div>'
    display(HTML(html_str))

In [41]:
matches = pl.read_csv('data/matches-with-ratings.csv')

In [42]:
last_n_matches = (
    matches
    .filter(
        pl.col('winner').str.to_lowercase().str.contains(player.lower()) |
        pl.col('loser').str.to_lowercase().str.contains(player.lower())
    )
    .sort('tournament_date', 'encounter_id', descending=True)
    .head(40)
)
losses = (
    last_n_matches
    .filter(pl.col('loser').str.to_lowercase().str.contains(player.lower()))
    .select(
        pl.col('winner').alias('tag'),
        'winner_rating',
        'tournament_name',
        'tournament_date',
        'loser_score',
    )
    .sort('winner_rating', descending=False)
)
wins = (
    last_n_matches
    .filter(pl.col('winner').str.to_lowercase().str.contains(player.lower()))
    .select(
        pl.col('loser').alias('tag'),
        'loser_rating',
        'tournament_name',
        'tournament_date',
        'winner_score',
    )
    .sort('loser_rating', descending=True)
)
display_side_by_side(
    losses.to_pandas(),
    wins.to_pandas(),
    titles=('Losses', 'Wins')
)

,tag,winner_rating,tournament_name,tournament_date,loser_score
0,Andrew,25.311602,Sunset Melee,2026-01-10,0
1,Swartzy,26.639584,New Game Plus Revival 9.9,2025-12-16,1
2,hc | keke,28.277297,Sunset Melee,2026-01-10,1
3,Gambit,30.787070,New Game Plus Revival 8.16,2025-08-05,1
4,hc | saucymain,33.203403,New Game Plus Revival 8.16,2025-08-05,0
5,Ant,35.249300,New Game Plus Revival 9.9,2025-12-16,1
,tag,loser_rating,tournament_name,tournament_date,winner_score
0,Badboi,30.384124,New Game Plus Revival 8.16,2025-08-05,2
1,Byrne,25.528163,New Game Plus Revival 9.9,2025-12-16,2
2,yungsos,22.377392,New Game Plus Revival 8.16,2025-08-05,2


# ELO of all attendees

In [43]:
# Use the tournament id from your earlier query
tournament_id = tournament_result['tournament']['id']

attendees_query = gql("""
query TournamentAttendees($tournamentId: ID!) {
  tournament(id: $tournamentId) {
    participants(query: {perPage: 512}) {
      nodes {
        id
        gamerTag
        user {
          id
          name
        }
      }
    }
  }
}
""")

attendees_variables = {"tournamentId": tournament_id}

try:
    attendees_result = client.execute(attendees_query, variable_values=attendees_variables)
    # Flatten and display as DataFrame
    import polars as pl
    attendee_nodes = attendees_result['tournament']['participants']['nodes']
    attendees = pl.DataFrame([
        {
            "id": a["id"],
            "gamerTag": a["gamerTag"],
            "user_id": a["user"]["id"] if a["user"] else None,
            "user_name": a["user"]["name"] if a["user"] else None,
        }
        for a in attendee_nodes
    ])
    attendees
except Exception as e:
    print("Error fetching tournament attendees:", e)

attendees

id,gamerTag,user_id,user_name
i64,str,i64,str
21080785,"""Duffy""",3166576,null
21082190,"""BonkCushy""",187,"""Dan Cushing"""
21078879,"""MEAT""",177939,"""Grant Foster"""
21083157,"""Cheezboom""",2762869,"""Jared B."""
21077283,"""Tpoz""",1001870,"""Pato Mora"""
21081837,"""sfy bees""",34163,"""Tyler Sherer"""
21082933,"""Lermonz""",614035,"""Lena Navin"""
21080263,"""Eol""",3213021,null
21076526,"""kwest""",106074,"""Karen West"""


In [44]:
# Join attendees on ratings and sort by rating (descending)
attendee_ratings = (
    attendees
    .with_columns(pl.col('gamerTag').str.to_lowercase().alias('gamertag'))
    .join(
        player_ratings.with_columns(
            pl.col('tag').str.split(' | ').list.last().str.to_lowercase()
        ),
        left_on='gamertag',
        right_on='tag',
        how='left'
    )
    .sort('rating', descending=True)
)
attendee_ratings

id,gamerTag,user_id,user_name,gamertag,url,rating
i64,str,i64,str,str,str,f64
21076526,"""kwest""",106074,"""Karen West""","""kwest""",null,null
21081400,"""Yep""",3313450,null,"""yep""",null,null
21082843,"""bonfire10""",409,"""Tori Paolillo""","""bonfire10""","""/league/nemelee/player/C9B8492…",42.254872
21081697,"""coffee""",548818,"""Alex S.""","""coffee""","""/league/nemelee/player/25479C6…",40.337837
21078879,"""MEAT""",177939,"""Grant Foster""","""meat""","""/league/nemelee/player/0D8C1F3…",34.761998
21078708,"""saucymain""",668760,null,"""saucymain""","""/league/nemelee/player/2407F41…",34.501203
21082190,"""BonkCushy""",187,"""Dan Cushing""","""bonkcushy""","""/league/nemelee/player/1056255…",30.466068
21042727,"""zaubermaus""",305778,"""Aaron Giera""","""zaubermaus""","""/league/nemelee/player/66DEE19…",29.000814
21082199,"""keke""",34096,"""Jake Donovan""","""keke""","""/league/nemelee/player/ACFD51E…",28.693111
